In [ ]:
import os 
import pandas as pd 
import numpy as np

from dotenv import load_dotenv

import requests 
import json 


TO DO:

- get list of air quality sensors (Golemio)
- get list of microclimate sensors (Golemio)
- get list of Weather stations (CHMI)

- join this to one dataset mapping the relevent locations together

- download relevant datasources, joint them based on mapping defined above 

- do analysis (...)

In [2]:
#data chmi

#https://opendata.chmi.cz/meteorology/climate/historical/data/1hour/

#kbely wsi 0-20000-0-11567

## this gets ony one specific station !!!


In [ ]:
chmi_url = 'https://opendata.chmi.cz/'
route = '/meteorology/climate/historical/metadata/meta1.json'

headers = {
    'accept': 'application/json',
    'User-Agent': 'JEM207 DataProcessingCourse (Educational access; contact: 19658413@fsv.cuni.cz)'
}

response = requests.get(f'{chmi_url}{route}', headers = headers, timeout = 60)

chmi_stations = response.json()

chmi_stations = chmi_stations['data']['data']

headers = chmi_stations['header'].split(',')
values = chmi_stations['values']

df_chmi_stat = pd.DataFrame(values, columns=headers)

df_chmi_stat = df_chmi_stat[df_chmi_stat['FULL_NAME'].str.contains('Praha', case=False, na=False)]

In [ ]:
df_chmi_stat["END_DATE_DT"] = pd.to_datetime(df_chmi_stat["END_DATE"], utc=True, errors="coerce")

df_chmi_stat = (
    df_chmi_stat.sort_values("END_DATE_DT")
    .drop_duplicates(subset="WSI", keep="last")
)

now_utc = pd.Timestamp.now(tz="UTC")
df_chmi_stat = df_chmi_stat[
    (df_chmi_stat["END_DATE_DT"] >= now_utc)
]

In [150]:
df_chmi_stat[df_chmi_stat['FULL_NAME']=='Praha, Klementinum']

,WSI,GH_ID,BEGIN_DATE,END_DATE,FULL_NAME,GEOGR1,GEOGR2,ELEVATION,END_DATE_DT
2217,0-203-0-11515,P1PKLM01,2023-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Klementinum",14.416436,50.086341,190.7,3999-12-31 23:59:00+00:00
2214,0-203-0-11514,P1PKLE01,2023-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Klementinum",14.416923,50.086634,190.7,3999-12-31 23:59:00+00:00


In [140]:
wsi_dict = dict(
    zip(
        df_chmi_stat["WSI"].astype(str),
        df_chmi_stat["FULL_NAME"].astype(str)  
    )
)

In [141]:
wsi_dict

{'0-203-0-11201020001': 'Praha, Vinohrady - Flora',
 '0-203-0-11515': 'Praha, Klementinum',
 '0-203-0-11105048001': 'Praha, Zadní Kopanina',
 '0-203-0-11201024001': 'Praha, Břevnov',
 '0-203-0-11101007001': 'Praha, Brdy',
 '0-203-0-10904013001': 'Praha, Komořany',
 '0-203-0-11202007001': 'Praha, Suchdol',
 '0-20000-0-11567': 'Praha, Kbely',
 '0-20000-0-11520': 'Praha, Libuš',
 '0-20000-0-11519': 'Praha, Karlov',
 '0-203-0-11514': 'Praha, Klementinum',
 '0-203-0-11201020003': 'Praha, Chodov',
 '0-20000-0-11518': 'Praha, Ruzyně'}

In [ ]:
### chmi data variables



In [ ]:
chmi_url = 'https://opendata.chmi.cz/'
route = '/meteorology/climate/historical/metadata/meta2.json'

headers = {
    'accept': 'application/json',
    'User-Agent': 'JEM207 DataProcessingCourse (Educational access; contact: 19658413@fsv.cuni.cz)'
}

response = requests.get(f'{chmi_url}{route}', headers = headers, timeout = 60)

chmi_vars = response.json()

chmi_vars = chmi_vars['data']['data']
headers = chmi_vars['header'].split(',')
values = chmi_vars['values']

df_chmi_vars = pd.DataFrame(values, columns=headers)

df_chmi_vars = df_chmi_vars[df_chmi_vars['WSI'].astype(str).isin(wsi_dict)]

In [157]:
df_chmi_vars[df_chmi_vars['EG_EL_ABBREVIATION'] == 'E']

,OBS_TYPE,WSI,BEGIN_DATE,END_DATE,EG_EL_ABBREVIATION,NAME,UN_DESCRIPTION,HEIGHT,SCHEDULE
58414,HLY,0-20000-0-11519,2022-09-01T00:00:00Z,3999-12-31T23:59:00Z,E,Tlak páry,hPa,1.99,1H


In [ ]:
# TODO dictionary of abbreviations and names

In [6]:
### golemio data 


##### air quality

## metadata: https://opendata.chmi.cz//air_quality/recent/metadata/metadata.json 

In [7]:
load_dotenv('api_key.env')
api_key = os.getenv('GOLEMIO_API_KEY')

print(api_key is not None)

True


In [8]:
api_url = 'https://api.golemio.cz/'
route = '/v2/airqualitystations'

headers = {
    'X-access-token': api_key, 
    'accept': 'application/json',
    'User-Agent': 'JEM207 DataProcessingCourse (Educational access; contact: 19658413@fsv.cuni.cz)'
}

params = {

}

response = requests.get(f'{api_url}{route}', headers = headers, timeout = 60)

data = response.json()

In [9]:
print(type(data))

print(data.keys())

<class 'dict'>
dict_keys(['features', 'type'])


In [10]:
print(json.dumps(data, indent=2, ensure_ascii = False))

{
  "features": [
    {
      "geometry": {
        "coordinates": [
          14.380116,
          50.084385
        ],
        "type": "Point"
      },
      "properties": {
        "measurement": {
          "AQ_hourly_index": "2B",
          "components": [
            {
              "averaged_time": {
                "averaged_hours": "3",
                "value": 24.7
              },
              "type": "NO2"
            },
            {
              "averaged_time": {
                "averaged_hours": "3",
                "value": 32.1
              },
              "type": "PM10"
            }
          ]
        },
        "id": "ABREA",
        "name": "Praha 6-Břevnov",
        "updated_at": "2026-04-28T10:45:00.710Z",
        "district": "praha-6"
      },
      "type": "Feature"
    },
    {
      "geometry": {
        "coordinates": [
          14.44365,
          50.108845
        ],
        "type": "Point"
      },
      "properties": {
        "measurement": {
   

In [11]:
def print_json_structure(obj, indent=0):
    pad = "  " * indent

    if isinstance(obj, dict):
        for key, value in obj.items():
            print(f"{pad}{key}: {type(value).__name__}")
            print_json_structure(value, indent + 1)

    elif isinstance(obj, list):
        print(f"{pad}[list] len={len(obj)}")
        if obj:
            print_json_structure(obj[0], indent + 1)

print_json_structure(data)

features: list
  [list] len=17
    geometry: dict
      coordinates: list
        [list] len=2
      type: str
    properties: dict
      measurement: dict
        AQ_hourly_index: str
        components: list
          [list] len=2
            averaged_time: dict
              averaged_hours: str
              value: float
            type: str
      id: str
      name: str
      updated_at: str
      district: str
    type: str
type: str


In [12]:
features = data['features']
df = pd.json_normalize(features)

In [13]:
tmp = df.explode("properties.measurement.components", ignore_index=True)
dirs = pd.json_normalize(
    tmp["properties.measurement.components"]
).add_prefix("properties.measurement.components.")

df = pd.concat([tmp.drop(columns=["properties.measurement.components"]), dirs], axis=1)

df.head()

,type,geometry.coordinates,geometry.type,properties.measurement.AQ_hourly_index,properties.id,properties.name,properties.updated_at,properties.district,properties.measurement.components.type,properties.measurement.components.averaged_time.averaged_hours,properties.measurement.components.averaged_time.value
0,Feature,"[14.380116, 50.084385]",Point,2B,ABREA,Praha 6-Břevnov,2026-04-28T10:45:00.710Z,praha-6,NO2,3,24.7
1,Feature,"[14.380116, 50.084385]",Point,2B,ABREA,Praha 6-Břevnov,2026-04-28T10:45:00.710Z,praha-6,PM10,3,32.1
2,Feature,"[14.44365, 50.108845]",Point,2B,AHOLA,Praha 7-Holešovice,2026-04-28T10:45:00.710Z,praha-7,NO2,3,33.2
3,Feature,"[14.44365, 50.108845]",Point,2B,AHOLA,Praha 7-Holešovice,2026-04-28T10:45:00.710Z,praha-7,PM10,3,40.1
4,Feature,"[14.44365, 50.108845]",Point,2B,AHOLA,Praha 7-Holešovice,2026-04-28T10:45:00.710Z,praha-7,PM2_5,3,23.4


In [14]:
station_cols = {
    'geometry.coordinates': 'coordinates', 
    'properties.id': 'id', 
    'properties.name': 'name', 
    'properties.district': 'district', 
    'properties.measurement.components.type': 'components'
}

In [15]:
air_quality_stations = df[station_cols.keys()]

air_quality_stations = air_quality_stations.rename(columns=station_cols)

In [16]:
air_quality_stations.head()

,coordinates,id,name,district,components
0,"[14.380116, 50.084385]",ABREA,Praha 6-Břevnov,praha-6,NO2
1,"[14.380116, 50.084385]",ABREA,Praha 6-Břevnov,praha-6,PM10
2,"[14.44365, 50.108845]",AHOLA,Praha 7-Holešovice,praha-7,NO2
3,"[14.44365, 50.108845]",AHOLA,Praha 7-Holešovice,praha-7,PM10
4,"[14.44365, 50.108845]",AHOLA,Praha 7-Holešovice,praha-7,PM2_5


In [17]:
air_quality_stations = (
    air_quality_stations.groupby('id', as_index=False)
    .agg({
        'coordinates': 'first',
        'name': 'first',
        'district': 'first',
        'components': list
    })
)

In [142]:
wsi_dict

{'0-203-0-11201020001': 'Praha, Vinohrady - Flora',
 '0-203-0-11515': 'Praha, Klementinum',
 '0-203-0-11105048001': 'Praha, Zadní Kopanina',
 '0-203-0-11201024001': 'Praha, Břevnov',
 '0-203-0-11101007001': 'Praha, Brdy',
 '0-203-0-10904013001': 'Praha, Komořany',
 '0-203-0-11202007001': 'Praha, Suchdol',
 '0-20000-0-11567': 'Praha, Kbely',
 '0-20000-0-11520': 'Praha, Libuš',
 '0-20000-0-11519': 'Praha, Karlov',
 '0-203-0-11514': 'Praha, Klementinum',
 '0-203-0-11201020003': 'Praha, Chodov',
 '0-20000-0-11518': 'Praha, Ruzyně'}

In [ ]:
### chmi weather data


In [152]:
base_url = "https://opendata.chmi.cz/"
route_template = "/meteorology/climate/historical/data/1hour/{year}/1h-{wsi}-{ym}.json"

url_header = {
    'accept': 'application/json',
    'User-Agent': 'JEM207 DataProcessingCourse (Educational access; contact: 19658413@fsv.cuni.cz)'
}

years = [2025]
months = [f'11'] #[f"{m:02d}" for m in range(1, 13)]

results = []

for wsi in wsi_dict:
    for year in years:
        for month in months:
            ym = f"{year}{month}"
            route = route_template.format(year=year, wsi=wsi, ym=ym)

            response = requests.get(
                f"{base_url}{route}",
                headers=url_header,
                timeout=60
            )
            if response.status_code == 200:
                data_response = response.json()
                data_response = data_response['data']['data']

                headers = data_response['header'].split(',')
                values = data_response['values']

                df_part = pd.DataFrame(values, columns=headers)
                df_part["WSI"] = wsi
                df_part["YEAR"] = year
                df_part["MONTH"] = month

                results.append(df_part)

            else:
                print(f"Failed for {wsi} {wsi_dict[wsi]} {year} {month}: {response.status_code}")


df_chmi = pd.concat(results, ignore_index=True)


Failed for 0-203-0-11515 Praha, Klementinum 2025 11: 404
Failed for 0-203-0-11105048001 Praha, Zadní Kopanina 2025 11: 404
Failed for 0-203-0-11101007001 Praha, Brdy 2025 11: 404
Failed for 0-203-0-11202007001 Praha, Suchdol 2025 11: 404
Failed for 0-203-0-11201020003 Praha, Chodov 2025 11: 404


In [153]:
df_chmi.head()

,STATION,ELEMENT,DT,VAL,FLAG,QUALITY,WSI,YEAR,MONTH
0,0-203-0-11201020001,E,2025-11-01T00:00:00Z,8.1,,0.0,0-203-0-11201020001,2025,11
1,0-203-0-11201020001,E,2025-11-01T01:00:00Z,7.9,,0.0,0-203-0-11201020001,2025,11
2,0-203-0-11201020001,E,2025-11-01T02:00:00Z,8.2,,0.0,0-203-0-11201020001,2025,11
3,0-203-0-11201020001,E,2025-11-01T03:00:00Z,7.8,,0.0,0-203-0-11201020001,2025,11
4,0-203-0-11201020001,E,2025-11-01T04:00:00Z,7.8,,0.0,0-203-0-11201020001,2025,11


In [18]:
#### golemio

### microclimate sensors

In [19]:
# TODO if necessary